In [1]:
# 一、准备文档
from langchain_text_splitters import MarkdownHeaderTextSplitter


def prepare_docs():
    # =============1.准备文档================
    with open('./resources/评估.md', 'r', encoding='utf-8') as f:
        doc = f.read()

    # =============2.切分文档================
    # 切分依据，这里是按照三级标题
    headers_to_split_on = [
        ("#", "h1"),
        ("##", "h2"),
        ("###", "h3"),
    ]

    # 创建切分器
    markdown_splitter = MarkdownHeaderTextSplitter(
        headers_to_split_on,
        strip_headers=False  # 在正文中是否去除标题，False，不去除
    )

    # 切分文档, 返回的是 list[Document]
    docs = markdown_splitter.split_text(doc)

    return docs

docs = prepare_docs()

In [2]:
# 二、初始化向量模型
from langchain_ollama import OllamaEmbeddings

# 1.初始化向量模型
embeddings = OllamaEmbeddings(
    model="qwen3-embedding:0.6b",  # 性价比高的模型
    dimensions=1024  # 可选：减少维度以节省存储
)

In [8]:
# 三、初始化Collection，或者叫VectorStore
from langchain_milvus import Milvus, BM25BuiltInFunction

vectorstore = Milvus(
    embeddings,                             # 向量模型
    collection_name="langchain_collection", # collection名称
    builtin_function=BM25BuiltInFunction(   # 生成稀疏向量的函数
        analyzer_params={"type": "chinese"} # 指定中文分词
    ),
    vector_field=["dense", "sparse"],       # 向量字段，包括稠密和稀疏
    connection_args={
        "uri": "http://192.168.150.101:19530",     # milvus的uri路径
    },
    # index_params=[index_param],             # 可选，索引参数，必须与vector_field对应
    drop_old=True,                          # 是否删除旧的collection，避免重复创建
)

In [9]:
# 四、添加数据
vectorstore.add_documents(docs)

[468439912009399512,
 468439912009399513,
 468439912009399514,
 468439912009399515,
 468439912009399516,
 468439912009399517,
 468439912009399518,
 468439912009399519,
 468439912009399520,
 468439912009399521,
 468439912009399522,
 468439912009399523,
 468439912009399524,
 468439912009399525,
 468439912009399526,
 468439912009399527,
 468439912009399528,
 468439912009399529,
 468439912009399530,
 468439912009399531,
 468439912009399532]

In [5]:
vectorstore.delete([468439912009399490])

True

In [10]:
# 五、加权融合混合检索

query = "给教师的建议是谁写的"

# 带得分的相似度检索，返回值是list[tuple[Document, float]]
result = vectorstore.similarity_search_with_score(
    query,
    ranker_type="weighted",
    ranker_params={"weights": [0.4, 0.6]}
)

for doc, score in result:
    print(f"============score: {score}==============")
    print(doc.model_dump_json(indent=2))

============score: 0.7107663154602051==============
{
  "id": null,
  "metadata": {
    "h2": "第一节 中外教育家及其教育思想",
    "h3": "（八）教育学分化时期代表人物及主要思想",
    "h1": "第一章 教育概述",
    "pk": 468439912009399519
  },
  "page_content": "### （八）教育学分化时期代表人物及主要思想  \n**马卡连柯**：著有《教育诗》，提出集体主义教育。  \n**克鲁普斯卡娅**：著有《国民教育与民主制度》，是最早以马克思主义为基础探讨教育问题的教育家。  \n**杨贤江**：著有《新教育大纲》，是中国第一部以马克思主义为指导的教育学著作。  \n**凯洛夫**：著有《教育学》，是世界第一部马克思主义的教育学著作。  \n**布鲁姆**：著有《教学目标分类学》，提出了掌握学习理论：所有学生都能学好；目标分为认知、情感、动作技能。  \n**布鲁纳**：著有《教学过程》，提出了结构主义教学理论，倡导发现式学习。  \n**瓦·根舍因**：著有《范例教学理论》，与布鲁纳和赞可夫被认为课程现代化的三大代表人物。  \n**赞可夫**：著有《教学与发展》，提出了发展性教学理论的五原则：高难度、高速度、理论知识起主导作用、理解学习过程、所有学生包括差生都得到发展的原则。  \n**苏霍姆林斯基**：著有《给教师的建议》《把整个心灵献给孩子》《帕夫雷什中学》，提出个性全面和谐发展的教育思想，他的著作被称为“活的教育学”。  \n**巴班斯基**：著有《教学过程最优化》《教学教育过程最优化》，把现代控制论、系统论观点用于教学论研究，提出教学过程最优化的理论。  \n**皮亚杰**：提出儿童认知发展阶段论、道德发展阶段论，以及儿童与环境互相作用的两个过程：同化、顺应。  \n**维果斯基**：提出心理学的“文化—历史”发展理论；最近发展区理论，教学必须走在发展的前面；支架式教学；“内化”学说。",
  "type": "Document"
}
============score: 0.4528707265853882==============
{
  "id": null,
  "meta

In [11]:
# 六、加权融合混合检索

query = "给教师的建议是谁写的"

# 带得分的相似度检索，返回值是list[tuple[Document, float]]
result = vectorstore.similarity_search_with_score(
    query,
    ranker_type="rrf",
    ranker_params={"k": 60}
)

for doc, score in result:
    print(f"============score: {score}==============")
    print(doc.model_dump_json(indent=2))

============score: 0.03226646035909653==============
{
  "id": null,
  "metadata": {
    "h1": "第一章 教育概述",
    "h2": "第一节 中外教育家及其教育思想",
    "h3": "（八）教育学分化时期代表人物及主要思想",
    "pk": 468439912009399519
  },
  "page_content": "### （八）教育学分化时期代表人物及主要思想  \n**马卡连柯**：著有《教育诗》，提出集体主义教育。  \n**克鲁普斯卡娅**：著有《国民教育与民主制度》，是最早以马克思主义为基础探讨教育问题的教育家。  \n**杨贤江**：著有《新教育大纲》，是中国第一部以马克思主义为指导的教育学著作。  \n**凯洛夫**：著有《教育学》，是世界第一部马克思主义的教育学著作。  \n**布鲁姆**：著有《教学目标分类学》，提出了掌握学习理论：所有学生都能学好；目标分为认知、情感、动作技能。  \n**布鲁纳**：著有《教学过程》，提出了结构主义教学理论，倡导发现式学习。  \n**瓦·根舍因**：著有《范例教学理论》，与布鲁纳和赞可夫被认为课程现代化的三大代表人物。  \n**赞可夫**：著有《教学与发展》，提出了发展性教学理论的五原则：高难度、高速度、理论知识起主导作用、理解学习过程、所有学生包括差生都得到发展的原则。  \n**苏霍姆林斯基**：著有《给教师的建议》《把整个心灵献给孩子》《帕夫雷什中学》，提出个性全面和谐发展的教育思想，他的著作被称为“活的教育学”。  \n**巴班斯基**：著有《教学过程最优化》《教学教育过程最优化》，把现代控制论、系统论观点用于教学论研究，提出教学过程最优化的理论。  \n**皮亚杰**：提出儿童认知发展阶段论、道德发展阶段论，以及儿童与环境互相作用的两个过程：同化、顺应。  \n**维果斯基**：提出心理学的“文化—历史”发展理论；最近发展区理论，教学必须走在发展的前面；支架式教学；“内化”学说。",
  "type": "Document"
}
============score: 0.016393441706895828==============
{
  "id": null,
  "m

In [13]:
# 七、过滤

query = "给教师的建议是谁写的"

# 带得分的相似度检索，返回值是list[tuple[Document, float]]
result = vectorstore.similarity_search_with_score(
    query,
    ranker_type="rrf",
    ranker_params={"k": 60},
    expr="h3 like '%代表人物%'"
)

for doc, score in result:
    print(f"============score: {score}==============")
    print(doc.model_dump_json(indent=2))

============score: 0.032786883413791656==============
{
  "id": null,
  "metadata": {
    "h2": "第一节 中外教育家及其教育思想",
    "h3": "（八）教育学分化时期代表人物及主要思想",
    "h1": "第一章 教育概述",
    "pk": 468439912009399519
  },
  "page_content": "### （八）教育学分化时期代表人物及主要思想  \n**马卡连柯**：著有《教育诗》，提出集体主义教育。  \n**克鲁普斯卡娅**：著有《国民教育与民主制度》，是最早以马克思主义为基础探讨教育问题的教育家。  \n**杨贤江**：著有《新教育大纲》，是中国第一部以马克思主义为指导的教育学著作。  \n**凯洛夫**：著有《教育学》，是世界第一部马克思主义的教育学著作。  \n**布鲁姆**：著有《教学目标分类学》，提出了掌握学习理论：所有学生都能学好；目标分为认知、情感、动作技能。  \n**布鲁纳**：著有《教学过程》，提出了结构主义教学理论，倡导发现式学习。  \n**瓦·根舍因**：著有《范例教学理论》，与布鲁纳和赞可夫被认为课程现代化的三大代表人物。  \n**赞可夫**：著有《教学与发展》，提出了发展性教学理论的五原则：高难度、高速度、理论知识起主导作用、理解学习过程、所有学生包括差生都得到发展的原则。  \n**苏霍姆林斯基**：著有《给教师的建议》《把整个心灵献给孩子》《帕夫雷什中学》，提出个性全面和谐发展的教育思想，他的著作被称为“活的教育学”。  \n**巴班斯基**：著有《教学过程最优化》《教学教育过程最优化》，把现代控制论、系统论观点用于教学论研究，提出教学过程最优化的理论。  \n**皮亚杰**：提出儿童认知发展阶段论、道德发展阶段论，以及儿童与环境互相作用的两个过程：同化、顺应。  \n**维果斯基**：提出心理学的“文化—历史”发展理论；最近发展区理论，教学必须走在发展的前面；支架式教学；“内化”学说。",
  "type": "Document"
}
============score: 0.032258063554763794==============
{
  "id": null,
  "

In [14]:
from pymilvus import Function, FunctionType

# 八、cross-encoder混合检索
def create_cross_encoder_ranker(queries: list[str]):
    return Function(
        name="dashscope_semantic_ranker",  # ranker名词，唯一即可
        input_field_names=["text"],  # 原始文档字段
        function_type=FunctionType.RERANK,  # ranker类型，这里是固定值
        params={
            "reranker": "model",  # rerank类型，基于模型rerank，也就是cross-encoder
            "provider": "ali",  # rerank模型提供者
            "model_name": "gte-rerank-v2",  # rerank模型名称
            "queries": queries,  # 查询条件
            "max_client_batch_size": 5,  # 向模型发送请求时的批处理限制
        },
    )

def cross_encoder_search(query: str, filter: str = None):
    # 1.创建 ranker
    ranker = create_cross_encoder_ranker([query])
    # 2.调用 similarity_search_with_score
    result = vectorstore.similarity_search_with_score(
        query,
        reranker=ranker,
        expr=filter
    )
    # 3.打印
    for doc, score in result:
        print(f"============score: {score}==============")
        print(doc.model_dump_json(indent=2))

In [15]:
cross_encoder_search("给教师的建议是谁写的？")

============score: 0.19992606341838837==============
{
  "id": null,
  "metadata": {
    "h1": "第一章 教育概述",
    "h2": "第一节 中外教育家及其教育思想",
    "h3": "（八）教育学分化时期代表人物及主要思想",
    "pk": 468439912009399519
  },
  "page_content": "### （八）教育学分化时期代表人物及主要思想  \n**马卡连柯**：著有《教育诗》，提出集体主义教育。  \n**克鲁普斯卡娅**：著有《国民教育与民主制度》，是最早以马克思主义为基础探讨教育问题的教育家。  \n**杨贤江**：著有《新教育大纲》，是中国第一部以马克思主义为指导的教育学著作。  \n**凯洛夫**：著有《教育学》，是世界第一部马克思主义的教育学著作。  \n**布鲁姆**：著有《教学目标分类学》，提出了掌握学习理论：所有学生都能学好；目标分为认知、情感、动作技能。  \n**布鲁纳**：著有《教学过程》，提出了结构主义教学理论，倡导发现式学习。  \n**瓦·根舍因**：著有《范例教学理论》，与布鲁纳和赞可夫被认为课程现代化的三大代表人物。  \n**赞可夫**：著有《教学与发展》，提出了发展性教学理论的五原则：高难度、高速度、理论知识起主导作用、理解学习过程、所有学生包括差生都得到发展的原则。  \n**苏霍姆林斯基**：著有《给教师的建议》《把整个心灵献给孩子》《帕夫雷什中学》，提出个性全面和谐发展的教育思想，他的著作被称为“活的教育学”。  \n**巴班斯基**：著有《教学过程最优化》《教学教育过程最优化》，把现代控制论、系统论观点用于教学论研究，提出教学过程最优化的理论。  \n**皮亚杰**：提出儿童认知发展阶段论、道德发展阶段论，以及儿童与环境互相作用的两个过程：同化、顺应。  \n**维果斯基**：提出心理学的“文化—历史”发展理论；最近发展区理论，教学必须走在发展的前面；支架式教学；“内化”学说。",
  "type": "Document"
}
============score: 0.12749579548835754==============
{
  "id": null,
  "me

In [16]:
cross_encoder_search("教育的负面作用是什么？")

============score: 0.290463387966156==============
{
  "id": null,
  "metadata": {
    "h1": "第二章 教育基本原理",
    "h2": "第一节 教育的功能",
    "h3": "（一）个体发展功能和社会发展功能",
    "pk": 468439912009399526
  },
  "page_content": "# 第二章 教育基本原理  \n## 第一节 教育的功能  \n### （一）个体发展功能和社会发展功能  \n教育的正向功能（积极功能）指教育有助于社会进步和个体发展的积极影响和作用。  \n教育的负向功能（消极功能）指阻碍社会进步和个体发展的消极影响和作用。  \n教育的显性功能是指教育活动依照教育目的，在实际运行中所出现的与之相吻合的结果。  \n教育的隐性功能指伴随显性功能所出现的非预期性的功能。",
  "type": "Document"
}
============score: 0.14794716238975525==============
{
  "id": null,
  "metadata": {
    "h1": "第二章 教育基本原理",
    "h2": "第二节 教育和社会的关系",
    "h3": "（一）政治与教育的关系",
    "pk": 468439912009399527
  },
  "page_content": "## 第二节 教育和社会的关系  \n### （一）政治与教育的关系  \n政治（经济制度）对教育具有决定作用，是决定教育性质的直接因素。具体表现为：决定了教育的性质和目的；决定了教育的领导权；决定了哪部分社会成员享有受教育的权利；决定了部分的教育内容；决定了教育的管理体制。  \n教育对政治发展的作用表现在：教育促进人的政治社会化；教育培养现代政治法律人才；教育促进现代政治民主化。",
  "type": "Document"
}
============score: 0.1413116455078125==============
{
  "id": null,
  "metadata": {
    "h1": "第二章 教育基本原理",
    "h2": "第二节 教育